# Thực hành Truy vấn SQL với PySpark
Notebook này thực hiện 10 loại truy vấn SQL trên tập dữ liệu Pakistan E-commerce sau khi đã được tiền xử lý.

In [29]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

# Khởi tạo Spark Session
spark = SparkSession.builder \
    .appName("Pakistan_Ecommerce_SQL") \
    .getOrCreate()

# Đọc dữ liệu ĐÃ ĐƯỢC TIỀN XỬ LÝ (từ file preprocessing.ipynb)
data_path = "hdfs://localhost:9000/user/hadoop/ecommerce/Pakistan_Ecommerce_Clean.parquet"

df = spark.read.parquet(data_path)

# TẠO TEMPORARY VIEW ĐỂ TRUY VẤN SQL
df.createOrReplaceTempView("ecommerce_data")
print("Đã tạo Temporary View: ecommerce_data với", df.count(), "dòng.")

Đã tạo Temporary View: ecommerce_data với 584238 dòng.


### 1. Truy vấn chọn lọc và trình chiếu cơ bản (Selection & Projection)
Sử dụng SELECT, FROM và WHERE để trích xuất các cột cụ thể và lọc các hàng thỏa mãn điều kiện.

In [30]:
query_1 = """
SELECT item_id, status, price, qty_ordered, grand_total, category_name_1 
FROM ecommerce_data 
WHERE price > 1000 AND status = 'complete'
"""
spark.sql(query_1).show(5)

+-------+--------+-------+-----------+-----------+-----------------+
|item_id|  status|  price|qty_ordered|grand_total|  category_name_1|
+-------+--------+-------+-----------+-----------+-----------------+
| 214247|complete| 2490.0|        1.0|     2490.0|mobiles & tablets|
| 219349|complete|16999.0|        1.0|    16999.0|mobiles & tablets|
| 223029|complete| 1193.5|        1.0|     1618.5|    home & living|
| 225650|complete| 8420.0|        1.0|     9495.0|               \n|
| 228454|complete| 1130.0|        1.0|     1130.0|       appliances|
+-------+--------+-------+-----------+-----------+-----------------+
only showing top 5 rows



### 2. Truy vấn kết nối (Join Queries)
Dùng để kết hợp dữ liệu từ hai hay nhiều bảng. Ở đây ta tạo một bảng phụ (bảng Mapping Danh mục) và Join vào dữ liệu chính.

In [31]:
# Tạo một Temporary View phụ để demo JOIN
category_data = [
    ("Women's Fashion", "Fashion"), 
    ("Beauty & Grooming", "Health & Beauty"), 
    ("Soghaat", "Gifts")
]
spark.createDataFrame(category_data, ["category_name_1", "category_group"]).createOrReplaceTempView("category_mapping")

query_2 = """
SELECT e.item_id, e.price, e.category_name_1, c.category_group
FROM ecommerce_data e
INNER JOIN category_mapping c ON lower(e.category_name_1) = lower(c.category_name_1)
WHERE e.status = 'complete'
"""
spark.sql(query_2).show(5)

+-------+------+---------------+--------------+
|item_id| price|category_name_1|category_group|
+-------+------+---------------+--------------+
| 881819|1000.0|women's fashion|       Fashion|
| 831776|8499.0|women's fashion|       Fashion|
| 830236|1670.0|women's fashion|       Fashion|
| 836518|1515.0|women's fashion|       Fashion|
| 832631|1875.0|women's fashion|       Fashion|
+-------+------+---------------+--------------+
only showing top 5 rows



### 3. Truy vấn gom nhóm và hàm tổng hợp (Group By & Aggregation)
Sử dụng GROUP BY và các hàm như COUNT, SUM, AVG.

In [32]:
query_3 = """
SELECT category_name_1, 
       COUNT(item_id) as total_orders, 
       SUM(grand_total) as total_revenue,
       AVG(price) as avg_price
FROM ecommerce_data
WHERE status = 'complete'
GROUP BY category_name_1
ORDER BY total_revenue DESC
"""
spark.sql(query_3).show(5)

+-----------------+------------+--------------------+------------------+
|  category_name_1|total_orders|       total_revenue|         avg_price|
+-----------------+------------+--------------------+------------------+
|mobiles & tablets|       41657| 4.889264502480011E8|11217.698514295318|
|       appliances|       20706|1.7851121896549997E8| 8657.059619433978|
|    entertainment|        9325|      1.5500216831E8| 18319.22289544236|
|  women's fashion|       23682| 8.765372514799999E7| 1708.236002449117|
|    men's fashion|       40288| 5.794101033999999E7| 758.5125173749005|
+-----------------+------------+--------------------+------------------+
only showing top 5 rows



### 4. Truy vấn con (Subqueries / Nested Queries)
Tìm các đơn hàng có giá trị lớn hơn giá trị trung bình của các đơn hàng thành công.

In [33]:
query_4 = """
SELECT item_id, price, grand_total, category_name_1
FROM ecommerce_data
WHERE price > (
    SELECT AVG(price) FROM ecommerce_data WHERE status = 'complete'
)
AND status = 'complete'
"""
spark.sql(query_4).show(5)

+-------+-------+-----------+-----------------+
|item_id|  price|grand_total|  category_name_1|
+-------+-------+-----------+-----------------+
| 219349|16999.0|    16999.0|mobiles & tablets|
| 225650| 8420.0|     9495.0|               \n|
| 231293|16899.0|    16899.0|mobiles & tablets|
| 237052|12599.0|    12599.0|               \n|
| 245281|12599.0|    12599.0|               \n|
+-------+-------+-----------+-----------------+
only showing top 5 rows



### 5. Truy vấn sử dụng Hàm cửa sổ (Window Functions)
Xếp hạng các đơn hàng theo doanh thu trong từng danh mục sử dụng RANK() OVER(PARTITION BY ... ORDER BY ...).

In [34]:
query_5 = """
SELECT item_id, category_name_1, grand_total,
       RANK() OVER (PARTITION BY category_name_1 ORDER BY grand_total DESC) as revenue_rank
FROM ecommerce_data
WHERE status = 'complete' AND grand_total IS NOT NULL
"""
spark.sql(query_5).show(10)

+-------+---------------+-----------+------------+
|item_id|category_name_1|grand_total|revenue_rank|
+-------+---------------+-----------+------------+
| 797146|     appliances|   353722.0|           1|
| 797144|     appliances|   353722.0|           1|
| 797145|     appliances|   353722.0|           1|
| 834655|     appliances|   215533.5|           4|
| 835599|     appliances|   193466.0|           5|
| 835431|     appliances|   193466.0|           5|
| 503404|     appliances|   174355.0|           7|
| 503405|     appliances|   174355.0|           7|
| 667938|     appliances|   101790.0|           9|
| 835697|     appliances|  101535.98|          10|
+-------+---------------+-----------+------------+
only showing top 10 rows



### 6. Truy vấn sử dụng Toán tử tập hợp (Set Operations)
Sử dụng UNION ALL để gộp đơn hàng từ 2 danh mục khác nhau.

In [35]:
query_6 = """
SELECT item_id, price, category_name_1
FROM ecommerce_data WHERE lower(category_name_1) = 'mobiles & tablets'

UNION ALL

SELECT item_id, price, category_name_1
FROM ecommerce_data WHERE lower(category_name_1) = 'appliances'
"""
spark.sql(query_6).show(5)

+-------+-------+-----------------+
|item_id|  price|  category_name_1|
+-------+-------+-----------------+
| 214247| 2490.0|mobiles & tablets|
| 219349|16999.0|mobiles & tablets|
| 229726|15199.0|mobiles & tablets|
| 230588|24999.0|mobiles & tablets|
| 231261|  375.0|mobiles & tablets|
+-------+-------+-----------------+
only showing top 5 rows



### 7. Truy vấn phân tích chuỗi thời gian (Time-Series Analysis)
Phân tích theo tháng/năm. Ở đây ta rút trích năm và tháng từ cột `created_at`.

In [36]:
query_7 = """
SELECT year(to_date(created_at, 'M/d/yyyy')) as order_year, 
       month(to_date(created_at, 'M/d/yyyy')) as order_month, 
       SUM(grand_total) as monthly_revenue,
       COUNT(item_id) as total_orders
FROM ecommerce_data
WHERE status = 'complete' AND created_at IS NOT NULL
GROUP BY year(to_date(created_at, 'M/d/yyyy')), month(to_date(created_at, 'M/d/yyyy'))
ORDER BY order_year DESC, order_month DESC
"""
spark.sql(query_7).show(5)

+----------+-----------+---------------+------------+
|order_year|order_month|monthly_revenue|total_orders|
+----------+-----------+---------------+------------+
|      2018|          8|        23001.0|          13|
|      2018|          7|       145529.0|          74|
|      2018|          6|        14129.0|          27|
|      2018|          5|2.19436986025E7|        1446|
|      2018|          4|  2.085664322E7|        5151|
+----------+-----------+---------------+------------+
only showing top 5 rows



### 8. Truy vấn sắp xếp và phân trang (Sorting & Pagination)
Sử dụng ORDER BY và LIMIT để lấy Top 10 đơn hàng có giá trị cao nhất.

In [37]:
query_8 = """
SELECT item_id, status, grand_total, category_name_1
FROM ecommerce_data
WHERE status = 'complete'
ORDER BY grand_total DESC
LIMIT 10
"""
spark.sql(query_8).show()

+-------+--------+-----------+-----------------+
|item_id|  status|grand_total|  category_name_1|
+-------+--------+-----------+-----------------+
| 508369|complete|   1.7888E7|mobiles & tablets|
| 508368|complete|   1.7888E7|mobiles & tablets|
| 451416|complete|  1039479.0|    entertainment|
| 416655|complete|   389459.0|mobiles & tablets|
| 416653|complete|   389459.0|mobiles & tablets|
| 797143|complete|   353722.0|    entertainment|
| 797146|complete|   353722.0|       appliances|
| 797144|complete|   353722.0|       appliances|
| 797145|complete|   353722.0|       appliances|
| 835121|complete|   293475.0|        computing|
+-------+--------+-----------+-----------------+



### 9. Truy vấn với Biểu thức Bảng Chung (CTE - Common Table Expressions)
Sử dụng WITH để định nghĩa bảng tạm tính tổng doanh thu mỗi danh mục, sau đó truy vấn từ CTE này.

In [38]:
query_9 = """
WITH CategoryRevenue AS (
    SELECT category_name_1, SUM(grand_total) as total_revenue
    FROM ecommerce_data
    WHERE status = 'complete'
    GROUP BY category_name_1
)
SELECT *
FROM CategoryRevenue
WHERE total_revenue > 1000000
ORDER BY total_revenue DESC
"""
spark.sql(query_9).show()

+------------------+--------------------+
|   category_name_1|       total_revenue|
+------------------+--------------------+
| mobiles & tablets| 4.889264502480011E8|
|        appliances|1.7851121896549997E8|
|     entertainment|      1.5500216831E8|
|   women's fashion| 8.765372514799999E7|
|     men's fashion| 5.794101033999999E7|
|         computing| 3.522981093500001E7|
|        superstore|3.3569570348500006E7|
| beauty & grooming|3.2159284933999993E7|
|     home & living|      2.3125941202E7|
|           soghaat|1.8068704700500004E7|
|                \n|     1.77903377715E7|
|            others|1.5720248790000003E7|
|   health & sports|     1.36341601525E7|
|       kids & baby|      1.1075011705E7|
|school & education|           2301800.3|
|             books|          1059447.92|
+------------------+--------------------+



### 10. Truy vấn thao tác cấu trúc (DDL) và dữ liệu (DML)
Thay vì UPDATE/DELETE trực tiếp (Spark SQL hạn chế trên file tĩnh), ta có thể sử dụng CREATE OR REPLACE TEMP VIEW để định nghĩa cấu trúc bảng mới từ dữ liệu.

In [39]:
query_10_ddl = """
CREATE OR REPLACE TEMP VIEW high_value_orders AS
SELECT item_id, grand_total, category_name_1
FROM ecommerce_data
WHERE grand_total > 50000
"""
spark.sql(query_10_ddl)
print("Đã tạo DDL View: high_value_orders")

query_10_select = "SELECT * FROM high_value_orders LIMIT 5"
spark.sql(query_10_select).show()

Đã tạo DDL View: high_value_orders
+-------+-----------+-----------------+
|item_id|grand_total|  category_name_1|
+-------+-----------+-----------------+
| 229214|   102582.0|    entertainment|
| 243597|   108999.0|        computing|
| 243892|    96499.0|mobiles & tablets|
| 246015|    76000.0|mobiles & tablets|
| 259436|    87499.0|mobiles & tablets|
+-------+-----------+-----------------+

